# Observed-outcome bilinear lasso experiment

This notebook fits the observed outcome stored in `y.npy`; it does **not** generate a simulated outcome or compare against true betas. CV chooses the penalty pair from validation MSE, then the final model is refit using all available subjects. Each run is saved in a new `model/results/real_y_<name>_<timestamp>/` folder.

To use another observed outcome, change `Y_FILE` in the configuration cell. `AGE_FILE` is read only because the shared data loader expects it; age is not used in fitting unless you explicitly select `age.npy` as `Y_FILE`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import pickle
import shutil
import sys
import uuid

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

MODEL_DIR = Path.cwd() / 'model'
if not (MODEL_DIR / 'bilinear_lasso.py').is_file():
    MODEL_DIR = Path.cwd()
if not (MODEL_DIR / 'parameters.json').is_file():
    raise FileNotFoundError('Start Jupyter in the repository root or model folder.')
sys.path.insert(0, str(MODEL_DIR))

from bilinear_lasso import bilinear_lasso
from config import PARAMS
from cross_validation import run_cv_over_lambdas
from data_utils import load_fmri_data
from evaluation import evaluate_prediction
from plot_utils import plot_fold_r2_by_lambda_pair, plot_lambda_heatmap


In [ ]:
EXPERIMENT_NAME = 'observed_y'
Y_FILE = 'y.npy'       # Change, for example, to 'age.npy' to fit age.
AGE_FILE = 'age.npy'  # Required by the shared loader; not used as a predictor.
RS_FILE = 'z_fc.npy'
EMO_FILE = 'z_sc.npy'
SEED = 2026
SAVE_CONNECTIVITY = True
FINAL_LAMBDA_PAIR = None  # None selects the lowest mean CV validation MSE.

RUN_PARAMETERS = dict(PARAMS)
# Example for a shorter exploratory run:
# RUN_PARAMETERS.update(LAMBDA1_GRID=[1, 5], LAMBDA2_GRID=[1, 5])

print('Outcome file:', Y_FILE)
print('Lambda pairs:', len(RUN_PARAMETERS['LAMBDA1_GRID']) * len(RUN_PARAMETERS['LAMBDA2_GRID']))
print('CV folds:', RUN_PARAMETERS['N_SPLITS'])

In [ ]:
X1, X2, y_observed, _ = load_fmri_data(
    data_dir=RUN_PARAMETERS['DATA_DIR'],
    p=RUN_PARAMETERS['P'],
    n_subjects=RUN_PARAMETERS['N_SUBJECTS'],
    rs_file=RS_FILE,
    emo_file=EMO_FILE,
    y_file=Y_FILE,
    age_file=AGE_FILE,
)
y_observed = np.asarray(y_observed, dtype=np.float64).ravel()
if not np.all(np.isfinite(y_observed)):
    raise ValueError('The observed outcome contains NaN or infinity.')
if np.std(y_observed) == 0:
    raise ValueError('The observed outcome must vary across subjects.')
print(f'Loaded {len(y_observed)} subjects; outcome mean={y_observed.mean():.4g}, SD={y_observed.std():.4g}')

In [ ]:
cv_results, cv_summary = run_cv_over_lambdas(
    X1, X2, y_observed,
    seed=SEED,
    lambda1_grid=RUN_PARAMETERS['LAMBDA1_GRID'],
    lambda2_grid=RUN_PARAMETERS['LAMBDA2_GRID'],
    n_jobs=RUN_PARAMETERS['N_JOBS'],
    n_splits=RUN_PARAMETERS['N_SPLITS'],
    max_iter=RUN_PARAMETERS['MAX_ITER'],
    step_size=RUN_PARAMETERS['STEP_SIZE'],
    tol=RUN_PARAMETERS['TOL'],
    num_candidates=RUN_PARAMETERS['NUM_CANDIDATES'],
    initialization=RUN_PARAMETERS['INITIALIZATION'],
    require_convergence=True,
)
display(cv_summary.sort_values(['Test MSE', 'lambda1', 'lambda2']))

In [ ]:
final_pair_was_explicit = FINAL_LAMBDA_PAIR is not None
if FINAL_LAMBDA_PAIR is None:
    selected = cv_summary.sort_values(['Test MSE', 'lambda1', 'lambda2']).iloc[0]
    FINAL_LAMBDA_PAIR = (float(selected['lambda1']), float(selected['lambda2']))
lambda1, lambda2 = FINAL_LAMBDA_PAIR
print(f'Selected final pair: lambda1={lambda1:g}, lambda2={lambda2:g}')

X1_mean = np.asarray(X1).mean(axis=2, keepdims=True)
X2_mean = np.asarray(X2).mean(axis=2, keepdims=True)
y_mean = float(y_observed.mean())
final_model = bilinear_lasso(X1-X1_mean, X2-X2_mean, y_observed-y_mean, lambda1, lambda2)
p = X1.shape[0]
final_model.initialize_beta(np.zeros((p, 1)), np.zeros((p, 1)))
final_model.fit(
    num_candidates=RUN_PARAMETERS['NUM_CANDIDATES'],
    max_iter=RUN_PARAMETERS['MAX_ITER'],
    step_size=RUN_PARAMETERS['STEP_SIZE'],
    tol=RUN_PARAMETERS['TOL'],
    disturbance=2.0,
    seed=SEED,
    initialization=RUN_PARAMETERS['INITIALIZATION'],
    require_convergence=True,
)
y_pred_centered = np.asarray(final_model.predict(final_model.beta1, final_model.beta2))
y_pred = y_pred_centered + y_mean
display(pd.DataFrame.from_dict(final_model.candidate_status, orient='index'))
print(f'Final fit stationarity: {final_model.stationarity_:.3g}')

In [ ]:
def json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f'Cannot serialize {type(value).__name__}')

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_dir = MODEL_DIR / 'results' / f'real_y_{EXPERIMENT_NAME}_{stamp}_{uuid.uuid4().hex[:6]}'
run_dir.mkdir(parents=True, exist_ok=False)
(run_dir / 'figures').mkdir()
metadata = {
    'schema_version': 1, 'status': 'complete', 'name': EXPERIMENT_NAME,
    'created_utc': stamp, 'seed': SEED, 'parameters': RUN_PARAMETERS,
    'outcome_file': Y_FILE, 'age_file_loaded_only': AGE_FILE,
    'final_lambda_pair': [lambda1, lambda2],
    'final_selection': 'explicit lambda pair' if final_pair_was_explicit else 'minimum mean CV validation MSE',
    'preprocessing': 'X2: log10(1+X2), normalize by transformed diagonal, zero diagonal; X1 unchanged.',
    'centering': 'CV uses training-fold means; final model uses full-sample means.',
}
(run_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2, default=json_default) + '\n')
data_to_save = {'y_observed': y_observed, 'y_centered': y_observed-y_mean, 'subject_index': np.arange(len(y_observed))}
if SAVE_CONNECTIVITY:
    data_to_save.update(X1=np.asarray(X1), X2=np.asarray(X2))
np.savez_compressed(run_dir / 'observed_data.npz', **data_to_save)
cv_summary.to_csv(run_dir / 'cv_summary.csv', index=False)
with (run_dir / 'cv_results.pkl').open('wb') as stream:
    pickle.dump(cv_results, stream, protocol=pickle.HIGHEST_PROTOCOL)
np.savez_compressed(
    run_dir / 'final_model.npz', beta1=np.asarray(final_model.beta1), beta2=np.asarray(final_model.beta2),
    y_true=y_observed, y_pred=y_pred, y_true_centered=y_observed-y_mean, y_pred_centered=y_pred_centered,
    X1_mean=X1_mean, X2_mean=X2_mean, y_mean=y_mean, lambda1=lambda1, lambda2=lambda2,
    selected_candidate=final_model.sel_idx, converged=final_model.converged_,
    stationarity=final_model.stationarity_, objective=final_model.l,
)
history = {'loss_history': final_model.loss_history, 'stationarity_history': final_model.stationarity_history,
           'trajectory': final_model.trajectory, 'candidate_status': final_model.candidate_status,
           'selected_candidate': final_model.sel_idx}
with (run_dir / 'model_history.pkl').open('wb') as stream:
    pickle.dump(history, stream, protocol=pickle.HIGHEST_PROTOCOL)
(run_dir / 'model_diagnostics.json').write_text(json.dumps(final_model.candidate_status, indent=2, default=json_default) + '\n')
for filename in ('bilinear_lasso.py', 'config.py', 'cross_validation.py', 'data_utils.py', 'evaluation.py', 'plot_utils.py'):
    shutil.copy2(MODEL_DIR / filename, run_dir / filename)
shutil.copy2(MODEL_DIR / 'parameters.json', run_dir / 'parameters.json')
print(f'Saved observed-outcome experiment to: {run_dir}')

In [ ]:
plot_lambda_heatmap(cv_summary, metric='Test R2', save_path=run_dir / 'figures' / 'cv_test_r2_heatmap.png')
plot_fold_r2_by_lambda_pair(cv_results, save_path=run_dir / 'figures' / 'cv_fold_r2.png')

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
roi = np.arange(p)
axes[0, 0].plot(roi, np.asarray(final_model.beta1).ravel())
axes[0, 0].axhline(0, color='black', linestyle=':')
axes[0, 0].set(title=r'Estimated $\beta_1$', xlabel='ROI index', ylabel='Coefficient')
axes[0, 1].plot(roi, np.asarray(final_model.beta2).ravel())
axes[0, 1].axhline(0, color='black', linestyle=':')
axes[0, 1].set(title=r'Estimated $\beta_2$', xlabel='ROI index', ylabel='Coefficient')
axes[1, 0].scatter(y_observed, y_pred, s=14, alpha=.6)
lo, hi = min(y_observed.min(), y_pred.min()), max(y_observed.max(), y_pred.max())
axes[1, 0].plot([lo, hi], [lo, hi], 'k--')
axes[1, 0].set(title='Observed versus fitted outcome (in-sample)', xlabel='Observed y', ylabel='Fitted y')
axes[1, 1].scatter(y_pred, y_observed-y_pred, s=14, alpha=.6)
axes[1, 1].axhline(0, color='black', linestyle='--')
axes[1, 1].set(title='Residuals', xlabel='Fitted y', ylabel='Observed - fitted')
fig.suptitle(f'{EXPERIMENT_NAME} | lambda1={lambda1:g}, lambda2={lambda2:g}')
fig.tight_layout()
fig.savefig(run_dir / 'figures' / 'final_fit_diagnostics.png', dpi=180, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for candidate, values in final_model.loss_history.items():
    axes[0].plot(values, label=f'Start {candidate+1}')
    axes[1].semilogy(np.maximum(final_model.stationarity_history[candidate], 1e-16), label=f'Start {candidate+1}')
axes[0].set(xlabel='Iteration', ylabel='Penalized objective')
axes[1].axhline(RUN_PARAMETERS['TOL'], color='black', linestyle='--', label='Tolerance')
axes[1].set(xlabel='Iteration', ylabel='L1 stationarity residual')
for ax in axes:
    ax.legend()
fig.tight_layout()
fig.savefig(run_dir / 'figures' / 'optimizer_diagnostics.png', dpi=180, bbox_inches='tight')
plt.show()

print('In-sample metrics:', evaluate_prediction(y_observed, y_pred))
print('Figures:', run_dir / 'figures')